# Advanced Kubeflow Pipelines: Scheduling and Automation

### What you will learn
- Pipeline scheduling best practices for production MLOps.
- How to implement automated retraining triggers based on model performance.
- Integrating KFP pipelines with external systems and events.

### Why this matters in MLOps
Automation is the cornerstone of MLOps. Manual retraining cycles introduce latency, human error, and inconsistency. Automated scheduling ensures models are continuously refreshed, monitored, and optimized — transforming ML from a one-time project into an ongoing operational capability.

### You're done when...
- A recurring pipeline schedule is created and submitted.
- A retraining trigger based on model performance threshold is implemented.
- KFP Schedule and RunTrigger configurations are understood.

## The MLOps Perspective
Scheduling transforms ad-hoc ML workflows into reliable, production systems. The "gold standard" in MLOps is a pipeline that schedules itself — retraining when new data arrives or when model performance degrades below a threshold. KFP's Schedule API and integration with external monitoring tools make this self-healing loop possible.

## Setup and Imports

In [ ]:
# Install requirements
!pip install -r requirements.txt

In [ ]:
# Import required libraries
import kfp
from kfp import dsl
from kfp.dsl import component, Schedule
from kfp import Client
from datetime import datetime, timedelta
import time

## Pipeline Scheduling Best Practices
Effective scheduling requires understanding trade-offs: frequent runs catch issues early but consume resources; infrequent runs save compute but may miss data drift. Best practices include:

- Aligning schedules with data arrival patterns (e.g., daily after ETL completes)
- Setting appropriate caching to avoid redundant computation
- Naming experiments and runs consistently for auditability
- Implementing idempotent components that can safely re-run

In [ ]:
# Define a reusable training pipeline for scheduling
@component(
    base_image="python:3.11",
    packages_to_install=["numpy", "pandas"]
)
def ingest_data(source: str, date: str) -> str:
    """Ingest data from source for a given date."""
    import pandas as pd
    import numpy as np
    df = pd.DataFrame({
        "feature_1": np.random.randn(100),
        "feature_2": np.random.randn(100),
        "target": np.random.randn(100)
    })
    output_path = f"/data/processed/{date}/dataset.csv"
    df.to_csv(output_path, index=False)
    print(f"Data ingested and saved to {output_path}")
    return output_path


@component(
    base_image="python:3.11",
    packages_to_install=["scikit-learn", "numpy"]
)
def train_and_evaluate(data_path: str, model_name: str) -> dict:
    """Train model and return evaluation metrics."""
    import numpy as np
    accuracy = float(np.random.uniform(0.80, 0.95))
    f1_score = float(np.random.uniform(0.75, 0.93))
    metrics = {
        "model_name": model_name,
        "accuracy": accuracy,
        "f1_score": f1_score,
        "data_path": data_path
    }
    print(f"Training complete: {metrics}")
    return metrics


@component(
    base_image="python:3.11"
)
def register_model(metrics: dict, registry_uri: str) -> str:
    """Register model in model registry."""
    model_version = f"v{datetime.now().strftime('%Y%m%d_%H%M%S')}"
    register_path = f"{registry_uri}/{metrics['model_name']}/{model_version}"
    print(f"Model registered at: {register_path}")
    return register_path

In [ ]:
# Define the pipeline that will be scheduled
@dsl.pipeline(
    name="daily-retraining-pipeline",
    description="Pipeline that runs daily to retrain and register models"
)
def daily_retraining_pipeline(
    data_source: str = "s3://data-bucket/latest",
    model_name: str = "production-model",
    registry_uri: str = "mlflow:///models"
):
    today = datetime.now().strftime("%Y-%m-%d")
    ingest_task = ingest_data(source=data_source, date=today)
    train_task = train_and_evaluate(
        data_path=ingest_task.output,
        model_name=model_name
    )
    register_task = register_model(
        metrics=train_task.output,
        registry_uri=registry_uri
    )

## Creating a Recurring Schedule
Use the KFP Schedule API to create recurring runs. The schedule can be configured with cron expressions, caching policies, and experiment names.

In [ ]:
# Create a recurring schedule via KFP client
# Note: Replace endpoint with your KFP deployment URL
kfp_endpoint = "http://ml-pipeline.kubeflow.svc.cluster.local:8888"
kfp_client = Client(host=kfp_endpoint)

# Define the schedule
experiment_name = "automated-retraining"
cron_expression = "0 2 * * *"  # Run daily at 2 AM

try:
    run = kfp_client.create_recurring_run(
        experiment_name=experiment_name,
        job_name="daily-retraining-job",
        cron_schedule=cron_expression,
        pipeline_package_path="./daily_retraining_pipeline.yaml",
        params={
            "data_source": "s3://data-bucket/latest",
            "model_name": "production-model",
            "registry_uri": "mlflow:///models"
        }
    )
    print(f"Recurring run created: {run}")
except Exception as e:
    print(f"Could not create recurring run (API may differ by version): {e}")
    print("\nAlternative: Use the DSL Schedule object instead.")

## Retraining Trigger Based on Model Performance
In production, you want to trigger retraining when model performance drops below a threshold. This combines monitoring metrics with conditional pipeline execution.

In [ ]:
# Define monitoring and retraining components
@component(
    base_image="python:3.11",
    packages_to_install=["numpy"]
)
def check_model_performance(model_uri: str, threshold: float) -> bool:
    """Check if model performance meets the threshold."""
    import numpy as np
    current_accuracy = float(np.random.uniform(0.70, 0.98))
    print(f"Current model accuracy: {current_accuracy:.4f}, threshold: {threshold}")
    return current_accuracy >= threshold


@component(
    base_image="python:3.11",
    packages_to_install=["numpy"]
)
def trigger_retraining(reason: str, current_model: str) -> str:
    """Generate retraining trigger payload."""
    import json
    payload = {
        "trigger_reason": reason,
        "current_model": current_model,
        "timestamp": datetime.now().isoformat(),
        "action": "retrain"
    }
    print(f"Retraining trigger: {json.dumps(payload, indent=2)}")
    return json.dumps(payload)

In [ ]:
# Define a performance-driven retraining pipeline
@dsl.pipeline(
    name="performance-driven-retraining",
    description="Pipeline that conditionally retrains based on model performance"
)
def performance_retraining_pipeline(
    model_uri: str = "models/production/latest",
    threshold: float = 0.85
):
    check_task = check_model_performance(model_uri=model_uri, threshold=threshold)

    with dsl.Condition(check_task.output == False, name="retrain-needed"):
        trigger_data = trigger_retraining(
            reason="performance_below_threshold",
            current_model=model_uri
        )
        print(f"Retraining required: {trigger_data.output}")

    with dsl.Condition(check_task.output == True, name="no-action-needed"):
        print("Model performance is acceptable, no retraining needed")

## Integration with External Systems
KFP can integrate with external systems for monitoring, alerting, and triggering. This includes:
- Sending webhook notifications when pipelines complete
- Writing pipeline status to external monitoring systems
- Triggering downstream processes (A/B testing, canary deployments)

In [ ]:
# Define a component that integrates with external monitoring
@component(
    base_image="python:3.11",
    packages_to_install=["requests"]
)
def notify_external_system(status: str, pipeline_name: str, metrics: dict) -> bool:
    """Send pipeline status notification to external system."""
    import requests
    import json
    notification = {
        "pipeline": pipeline_name,
        "status": status,
        "metrics": metrics,
        "timestamp": datetime.now().isoformat()
    }
    print(f"Notification payload: {json.dumps(notification, indent=2)}")
    return True


@component(
    base_image="python:3.11",
    packages_to_install=["requests"]
)
def trigger_downstream_pipeline(upstream_metrics: dict, pipeline_endpoint: str) -> str:
    """Trigger a downstream pipeline after successful upstream run."""
    import json
    trigger_id = f"downstream-{datetime.now().strftime('%Y%m%d_%H%M%S')}"
    payload = {
        "trigger_id": trigger_id,
        "pipeline_endpoint": pipeline_endpoint,
        "metrics": upstream_metrics
    }
    print(f"Downstream trigger: {json.dumps(payload, indent=2)}")
    return trigger_id

In [ ]:
# Pipeline with external system integration
@dsl.pipeline(
    name="integrated-pipeline",
    description="Pipeline with external monitoring and downstream triggers"
)
def integrated_pipeline(
    data_source: str = "s3://data/latest",
    notify_url: str = "https://monitor.example.com/api/events"
):
    train_metrics = train_and_evaluate(
        data_path=data_source,
        model_name="integrated-model"
    )
    notify_task = notify_external_system(
        status="completed",
        pipeline_name="integrated-pipeline",
        metrics=train_metrics.output
    )
    downstream_task = trigger_downstream_pipeline(
        upstream_metrics=train_metrics.output,
        pipeline_endpoint="https://kfp.example.com/pipelines/deploy/v1"
    )

## Fill-in-the-Blank Exercises

In [ ]:
# Exercise 1: Complete the cron expression for a schedule that runs
# every 6 hours

schedule_config = Schedule(
    cron_expression="___",
    pipeline=daily_retraining_pipeline,
    experiment_name="frequent-retraining"
)
print(f"Schedule configured with: {schedule_config}")
# Hint: Every 6 hours means running at minute 0 of hours 0, 6, 12, 18

In [ ]:
# Exercise 2: Complete the retraining trigger condition
# Trigger retraining when f1_score drops below 0.80

def should_retrain(___):
    """Determine if retraining is needed based on metrics."""
    threshold = 0.80
    needs_retrain = metrics.get("f1_score", 0) < ___
    print(f"F1 Score: {metrics.get('f1_score', 0):.3f}, Retrain: {needs_retrain}")
    return needs_retrain

# Test with sample metrics
sample_metrics = {"accuracy": 0.82, "f1_score": 0.78}
result = should_retrain(sample_metrics)
print(f"Should retrain: {result}")
# Hint: The function parameter should receive the metrics dictionary

## Summary
In this notebook, you learned:
- Pipeline scheduling best practices for production MLOps
- Creating recurring pipeline schedules with KFP Schedule API
- Implementing performance-driven retraining triggers
- Integrating KFP pipelines with external monitoring and downstream systems